In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
flarndsim2 = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250606/larndsim_waveforms_unipolar_center.npz')
flarndsim = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250611/larndsim_waveforms_unipolar_center_long_twindow.npz')
fres = np.load('/home/yousen/Public/ndlar_shared/data/responses/response_v2a_center_unipolar_30p431cm_0p050us_bin_size0p04434.npz')

In [ ]:
ftred = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250609/single_segment_for_larndsim_unipolar_center.npz')
ftred = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250615/single_segment_for_larndsim_unipolar_center_30cm.npz')

In [ ]:
flarndsim2['charge_samplings'][0,0,0,0,:,2] - ftred['tpc_cathode_tpc2']

In [ ]:
def id2pixel(pid):
    """
    Convert the unique pixel identifer to an x,y,plane tuple

    Args:
        pid (int): unique pixel identifier
    Returns:
        tuple: number of pixel pitches in x-dimension,
            number of pixel pitches in y-dimension,
            pixel plane number
    """
    npixels = (2*70, 4*70)
    return (pid % npixels[0], (pid // npixels[0]) % npixels[1],
            (pid // (npixels[0] * npixels[1])))

id2pixel(flarndsim['unique_pixel_index'][0,4])

In [ ]:
vdrift = 0.1596452482154287
t0 = flarndsim['segment_info']['t0'][0]
pix_ind = 4
z = (flarndsim2['charge_samplings'][0,0,0,0,:,2] - ftred['tpc_cathode_tpc2'])
y = (flarndsim2['charge_samplings'][0,0,0,0,:,1] - ftred['tpc_lower_left_tpc2'][0])/0.4434
x = (flarndsim2['charge_samplings'][0,0,0,0,:,0] - ftred['tpc_lower_left_tpc2'][1])/0.4434
q = flarndsim2['charge_samplings'][0,0,0,0,:,-1]
m = (y<235) &(y>=234) & (x<101) & (x>=100)

t, edges = np.histogram(z[m]/vdrift + t0, range=(-83, -81), bins=40, weights = q[m])

In [ ]:
plt.bar(edges[:-1], t, width=np.diff(edges))

In [ ]:
induce_current = np.convolve(t, fres['response'][0,0])[1:].reshape(-1, 2).sum(axis=-1)*0.05

ttred = ftred['current_tpc2_batch0_location'][1,2]//2+np.arange(len(ftred['current_tpc2_batch0'][0, 0, 0, ::2])) + 0.05
tlarndsim = np.arange(len(flarndsim['pixel_waveform'][0,4,::]))
tred_args = np.argsort(np.sum(np.squeeze(ftred['current_tpc2_batch0']), axis=-1))[::-1]
larndsim_args = np.argsort(np.sum(np.squeeze(flarndsim['pixel_waveform']), axis=-1))[::-1]
ctred = ftred['current_tpc2_batch0'][tred_args][:2].reshape(2, -1)
clarnd = np.squeeze(flarndsim['pixel_waveform'])[larndsim_args][:2].reshape(2, -1)

tlarndsim2 = np.arange(len(flarndsim2['pixel_waveform'][0,4,::]))
larndsim2_args = np.argsort(np.sum(np.squeeze(flarndsim2['pixel_waveform']), axis=-1))[::-1]
clarnd2 = np.squeeze(flarndsim2['pixel_waveform'])[larndsim2_args][:2].reshape(2, -1)
plt.plot(tlarndsim2, 0.1*clarnd2[0], label='larnd-sim (short window) leading pixel')
plt.plot(tlarndsim, 0.1*clarnd[0], label='larnd-sim (long window) leading pixel', linestyle='--')

plt.plot(edges[0]//0.1+0.5+np.arange(len(induce_current)), induce_current, linestyle='-.')
plt.xlim(1000, 1100)

In [ ]:
t, edges = np.histogram(z[m]/vdrift + t0, range=(-83, -81), bins=40, weights = q[m])

In [ ]:
np.sum(induce_current*0.1), np.sum(t), np.sum(q[m])

In [ ]:
plt.hist(y, weights=q)

In [ ]:
np.sum(m)/500_000

# Effective charge related

In [ ]:
ftred['effq_tpc2_batch0'].shape, ftred['effq_tpc2_batch0_location'].shape

In [ ]:
uit = np.unique(ftred['effq_tpc2_batch0_location'][:,2])
effq = np.zeros(len(uit))
for i, it in enumerate(uit):
    m = it == ftred['effq_tpc2_batch0_location'][:,2]
    print(np.sum(m))
    effq[i] = np.sum(ftred['effq_tpc2_batch0'][:,-1][m])

In [ ]:
plt.plot(uit, effq)
print(uit)

In [ ]:
ic2 = np.convolve(effq*np.sum(t)/np.sum(q), fres['response'][0,0][:])[:-1] * 0.05
tic2 = uit[0] + np.arange(len(ic2)) - fres['drift_length']/vdrift//0.05 + 20 - 5.5
plt.plot(tic2[::2]//2, ic2.reshape(-1,2).sum(axis=-1), label='effq convo')

ttred = ftred['current_tpc2_batch0_location'][1,2]//2+np.arange(len(ftred['current_tpc2_batch0'][0, 0, 0, ::2])) + 0.05
tlarndsim = np.arange(len(flarndsim['pixel_waveform'][0,4,::]))
tred_args = np.argsort(np.sum(np.squeeze(ftred['current_tpc2_batch0']), axis=-1))[::-1]
larndsim_args = np.argsort(np.sum(np.squeeze(flarndsim['pixel_waveform']), axis=-1))[::-1]
ctred = ftred['current_tpc2_batch0'][tred_args][:2].reshape(2, -1)
clarnd = np.squeeze(flarndsim['pixel_waveform'])[larndsim_args][:2].reshape(2, -1)

tlarndsim2 = np.arange(len(flarndsim2['pixel_waveform'][0,4,::]))
larndsim2_args = np.argsort(np.sum(np.squeeze(flarndsim2['pixel_waveform']), axis=-1))[::-1]
clarnd2 = np.squeeze(flarndsim2['pixel_waveform'])[larndsim2_args][:2].reshape(2, -1)
plt.plot(tlarndsim2, 0.1*clarnd2[0], label='larnd-sim (short window) leading pixel')
plt.plot(tlarndsim, 0.1*clarnd[0], label='larnd-sim (long window) leading pixel', linestyle='--')

plt.plot(edges[0]//0.1+0.5+np.arange(len(induce_current)), induce_current, linestyle='-.')
plt.xlim(1000, 1100)

In [ ]:
(19.250494-3.079)/0.1596452482154287/0.05 + t0/0.05, uit[0], 3 * flarndsim2['segment_info']['long_diff']/vdrift/0.05

In [ ]:
((30.431-2500*vdrift*0.05)-10.431)/vdrift/0.05

In [ ]:
ftred['tpc_anode_tpc2'], flarndsim2['segment_info']['z_start'], flarndsim2['segment_info']['z_end'], flarndsim2['segment_info']['long_diff']

In [ ]:
ic3 = np.convolve(effq*np.sum(t)/np.sum(q), fres['response'][0,0][:])[:-1] * 0.05
tic3 = np.rint((19.250494-2.91)/0.1596452482154287/0.05).astype(int) + t0/0.05 + np.arange(len(ic3)) - fres['drift_length']/vdrift//0.05
# tic3 = np.rint(((19.250494+19.282745)/2-3.079)/0.1596452482154287/0.05).astype(int) + t0/0.05 + np.arange(len(ic3)) - fres['drift_length']/vdrift//0.05
# tic3 = np.rint((19.282745-2.91)/0.1596452482154287/0.05).astype(int) + t0/0.05 + np.arange(len(ic3)) - fres['drift_length']/vdrift//0.05

plt.plot(tic2[::2]//2 + 3, ic2.reshape(-1,2).sum(axis=-1), label='effq convo, t from diffused') # 3 is from difference between vdrift and 0.16
plt.plot(tic3[::2]//2, ic3.reshape(-1,2).sum(axis=-1), label='effq convo, t from end points')

ttred = ftred['current_tpc2_batch0_location'][1,2]//2+np.arange(len(ftred['current_tpc2_batch0'][0, 0, 0, ::2])) + 0.05
tlarndsim = np.arange(len(flarndsim['pixel_waveform'][0,4,::]))
tred_args = np.argsort(np.sum(np.squeeze(ftred['current_tpc2_batch0']), axis=-1))[::-1]
larndsim_args = np.argsort(np.sum(np.squeeze(flarndsim['pixel_waveform']), axis=-1))[::-1]
ctred = ftred['current_tpc2_batch0'][tred_args][:2].reshape(2, -1)
clarnd = np.squeeze(flarndsim['pixel_waveform'])[larndsim_args][:2].reshape(2, -1)

tlarndsim2 = np.arange(len(flarndsim2['pixel_waveform'][0,4,::]))
larndsim2_args = np.argsort(np.sum(np.squeeze(flarndsim2['pixel_waveform']), axis=-1))[::-1]
clarnd2 = np.squeeze(flarndsim2['pixel_waveform'])[larndsim2_args][:2].reshape(2, -1)
plt.plot(tlarndsim2, 0.1*clarnd2[0], label='larnd-sim (short window) leading pixel')
plt.plot(tlarndsim, 0.1*clarnd[0], label='larnd-sim (long window) leading pixel', linestyle='--')

plt.plot(edges[0]//0.1+0.5+np.arange(len(induce_current)), induce_current, linestyle='-.')
plt.xlim(1000, 1100)
plt.legend()
plt.grid()

In [ ]:
fres['drift_length']/vdrift, np.max(np.nonzero(fres['response'][0,0])[0])*0.05

In [ ]:
(190.55-190.616)//0.05

In [ ]:
fres['response'][0,0][np.max(np.nonzero(fres['response'][0,0])[0])], np.max(np.nonzero(fres['response'][0,0])[0])

In [ ]:
fres['response'][0,0][np.argmax(fres['response'][0,0])], np.argmax(fres['response'][0,0])

In [ ]:
np.max(np.nonzero(flarndsim['pixel_waveform'][0,4])[0])

In [ ]:
(1186*-6.08)*vdrift - (19.282745-3.079)

In [ ]:
10.43/vdrift

In [ ]:
((3811-2500)*0.05 - 10.43/vdrift)/0.05

In [ ]:
flarndsim2['segment_info']['z_start'], flarndsim2['segment_info']['z_end']

In [ ]:
(flarndsim2['segment_info']['z_start'] - ftred['tpc_cathode_tpc2'])/vdrift/0.05, (flarndsim2['segment_info']['z_end'] - ftred['tpc_cathode_tpc2'])/vdrift/0.05

In [ ]:
(3810-1765)*0.05 * 10 , (3810-1761)*0.05 * 10 , flarndsim2['segment_info']['t0']/0.1

In [ ]:
np.argmax(clarnd2[0]),  np.argmax(clarnd2[0]) - 61 - 1023, clarnd2[0][1080:1086]

In [ ]:
fres['response'][0,0,3805:3815], len(fres['response'][0,0,3805:3815])

In [ ]:

(3810)*0.05/0.05, fres['drift_length']/vdrift//0.05, ((3810-2500)*0.05/0.05 - 10.475/vdrift//0.05)*0.05

In [ ]:
flarndsim2['segment_info']['t_end']

In [ ]:
fres['drift_length']-(2500*vdrift*0.05), 2500*(vdrift-0.16)/vdrift

In [ ]:
ftred = np.load("/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250612/single_segment_for_larndsim_unipolar_center_30cm.npz", allow_pickle=True)
ftred = np.load("/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250615/single_segment_for_larndsim_unipolar_center_30cm.npz", allow_pickle=True)


print(ftred['current_tpc2_batch0_location'])
ttred = ftred['current_tpc2_batch0_location'][1,2]/2+np.arange(len(ftred['current_tpc2_batch0'][0, 0, 0, ::2]))
# print(ttred[0], ttred[-1], print(ftred['current_tpc2_batch0'])
tlarndsim = np.arange(len(flarndsim['pixel_waveform'][0,4,::]))
tred_args = np.argsort(np.sum(np.squeeze(ftred['current_tpc2_batch0']), axis=-1))[::-1]
larndsim_args = np.argsort(np.sum(np.squeeze(flarndsim['pixel_waveform']), axis=-1))[::-1]
ctred = ftred['current_tpc2_batch0'][tred_args][:2].reshape(2, -1)
clarnd = np.squeeze(flarndsim['pixel_waveform'])[larndsim_args][:2].reshape(2, -1)

tlarndsim2 = np.arange(len(flarndsim2['pixel_waveform'][0,4,::]))
larndsim2_args = np.argsort(np.sum(np.squeeze(flarndsim2['pixel_waveform']), axis=-1))[::-1]
clarnd2 = np.squeeze(flarndsim2['pixel_waveform'])[larndsim2_args][:2].reshape(2, -1)

plt.plot(ttred, 1000*ctred[0].reshape(-1,2).sum(axis=-1), label='tred leading pixel')
plt.plot(ttred, 1000*ctred[1].reshape(-1,2).sum(axis=-1), label='tred subleading pixel')
plt.plot(tlarndsim, 0.1*clarnd[0], label='larnd-sim (long window) leading pixel', linestyle='--')
plt.plot(tlarndsim, 0.1*clarnd[1], label='larnd-sim (long window) subleading pixel', linestyle='--')
plt.plot(tlarndsim2, 0.1*clarnd2[0], label='larnd-sim (short window) leading pixel', linestyle='--')
plt.plot(tlarndsim2, 0.1*clarnd2[1], label='larnd-sim (short window) subleading pixel', linestyle='--')
plt.legend()
plt.xlim(1000, 1100)
# plt.xlim(1000, 1500)
plt.grid(True)

print(clarnd.shape, clarnd2.shape)
print(clarnd[0][1087:1087+10])
print(np.sum(ctred[0]), np.sum(ctred[1]))
print(np.sum(ftred['current_tpc2_batch0']))

In [ ]:
2500*(0.16-vdrift)/vdrift

In [ ]:
np.rint((19.250494-3.079)/0.1596452482154287/0.05).astype(int) / 2, np.rint((19.250494-3.079)/0.1596452482154287/0.05).astype(int) / 2, ftred['current_tpc2_batch0_location'][1,2]-3

In [ ]:
fres2 = np.load("/home/yousen/Public/ndlar_shared/data/responses/response_v2a_center_unipolar_10p431cm_0p050us_bin_size0p04434.npz")
fres2['drift_length']

In [ ]:
ftred['drtoa'], np.sum(ftred['effq_tpc2_batch0'][:,-1])

In [ ]:
ftred['tpc_cathode_tpc2'], ftred['tpc_anode_tpc2'], 33.34125-2.9102500000000013
# (63.93+3.07)/2

In [ ]:
-30.431)/vdrift

In [ ]:
ftred['tpc_cathode_tpc2'] - ftred['tpc_anode_tpc2'], fres['drift_length'], (30.431-30.272250000000003)/vdrift/0.1, (3.07-2.91)/vdrift/0.05

In [ ]:
flarndsim2['segment_info']['t_end'], flarndsim2['segment_info']['t0'], flarndsim2['segment_info']['t_end'] - flarndsim2['segment_info']['t0'], (flarndsim2['segment_info']['z_start'] - 2.91)/vdrift, (flarndsim2['segment_info']['z_end'] - 2.91)/vdrift, (flarndsim2['segment_info']['z_start'] - ftred['tpc_anode_tpc2'])/vdrift, (flarndsim2['segment_info']['z_end'] - ftred['tpc_anode_tpc2'])/vdrift, 

In [ ]:
ftred['current_tpc2_batch0_location']

In [ ]:
np.min(ftred['effq_tpc2_batch0_location'][:,-1])

In [ ]:
uqp = np.unique(ftred['effq_tpc2_batch0_location'][:,:2], axis=0)
for pix in uqp:
    m = ftred['effq_tpc2_batch0_location'][:,:2] == pix[None,:]
    m = m.all(axis=1)
    l = ftred['effq_tpc2_batch0_location'][:,-1][m] - int(fres['drift_length']/vdrift) + fres['drift_length']/vdrift
    q = ftred['effq_tpc2_batch0'][:,-1][m]
    plt.plot(l*0.05, q, 'o-', label=f'effq on pixel {pix.tolist()} (TRED)')

yvals = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,1]-ftred['tpc_lower_left_tpc2'][0])/0.4434
zvals = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,0]-ftred['tpc_lower_left_tpc2'][1])/0.4434

m1 = (yvals < 235) & (yvals >= 234) & (zvals >= 100 ) & (zvals < 101)
q1 = flarndsim2['charge_samplings'][0, 0, 1, 0,:,-1][m1]
x1 = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,2][m1]-ftred['tpc_anode_tpc2'])/vdrift + t0
_, _, _ = plt.hist(x1, range=(107, 110), bins=60, weights=q1, alpha=0.5, label='q on pixel (234, 100) (larndsim)')

m2 = (yvals < 236) & (yvals >= 235) & (zvals >= 100 ) & (zvals < 101)
q2 = flarndsim2['charge_samplings'][0, 0, 1, 0,:,-1][m2]
x2 = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,2][m2]-ftred['tpc_anode_tpc2'])/vdrift + t0
print(ftred['tpc_anode_tpc2'])
_, _, _ = plt.hist(x2, range=(107, 110), bins=60, weights=q2, alpha=0.5, label='q on pixel (235, 100) (larndsim)')
plt.legend()
plt.grid(True)

In [ ]:
(ftred['global_tref_tpc2_batch0'][1] - t0)/0.05

In [ ]:
(flarndsim2['segment_info']['z_end'] - ftred['tpc_anode_tpc2'])/vdrift + t0

In [ ]:
(flarndsim2['segment_info']['z_start'] - ftred['tpc_anode_tpc2'])/vdrift + t0

In [ ]:
(108.4393371+108.64135602)/2, ftred['tpc_anode_tpc2'], ftred['tpc_cathode_tpc2']

In [ ]:
(flarndsim2['segment_info']['z_start'] - ftred['tpc_cathode_tpc2'])/vdrift + t0, (flarndsim2['segment_info']['z_end'] - ftred['tpc_cathode_tpc2'])/vdrift + t0

In [ ]:
uqp = np.unique(ftred['effq_tpc2_batch0_location'][:,:2], axis=0)
for pix in uqp:
    m = ftred['effq_tpc2_batch0_location'][:,:2] == pix[None,:]
    m = m.all(axis=1)
    l = ftred['effq_tpc2_batch0_location'][:,-1][m] - int(fres['drift_length']/vdrift) + fres['drift_length']/vdrift
    q = ftred['effq_tpc2_batch0'][:,-1][m]
    plt.plot(l*0.05, q, 'o-', label=f'effq on pixel {pix.tolist()} (TRED)')

yvals = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,1]-ftred['tpc_lower_left_tpc2'][0])/0.4434
zvals = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,0]-ftred['tpc_lower_left_tpc2'][1])/0.4434

m1 = (yvals < 235) & (yvals >= 234) & (zvals >= 100 ) & (zvals < 101)
q1 = flarndsim2['charge_samplings'][0, 0, 1, 0,:,-1][m1]
x1 = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,2][m1]-ftred['tpc_anode_tpc2'])/vdrift + t0
_, _, _ = plt.hist(yvals, range=(107, 110), bins=60, weights=q1, alpha=0.5, label='q on pixel (234, 100) (larndsim)')

m2 = (yvals < 236) & (yvals >= 235) & (zvals >= 100 ) & (zvals < 101)
q2 = flarndsim2['charge_samplings'][0, 0, 1, 0,:,-1][m2]
x2 = (flarndsim2['charge_samplings'][0, 0, 1, 0,:,2][m2]-ftred['tpc_anode_tpc2'])/vdrift + t0
print(ftred['tpc_anode_tpc2'])
_, _, _ = plt.hist(yvals[m1], range=(107, 110), bins=60, weights=q2, alpha=0.5, label='q on pixel (235, 100) (larndsim)')
plt.legend()
plt.grid(True)